In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
df = pd.read_csv('alphatrade_phase1.csv')

print(df.shape)
df.head()

(1508, 19)


,Date,Close,High,Low,Open,Volume,SMA_20,SMA_50,EMA_20,EMA_50,Daily_Return,Volatility,RSI,MACD,Signal_Line,BB_Middle,BB_Upper,BB_Lower,Signal
0,2020-01-02,72.333878,72.394086,71.091184,71.344054,135480400,NaN,NaN,72.333878,72.333878,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0
1,2020-01-03,71.630630,72.389250,71.406659,71.563198,146322800,NaN,NaN,72.266902,72.306299,-0.009722,NaN,NaN,-0.056099,-0.011220,NaN,NaN,NaN,0
2,2020-01-06,72.201424,72.239958,70.503561,70.754028,118387200,NaN,NaN,72.260666,72.302186,0.007969,NaN,NaN,-0.053879,-0.019752,NaN,NaN,NaN,0
3,2020-01-07,71.861847,72.466330,71.642689,72.211049,108872000,NaN,NaN,72.222683,72.284918,-0.004703,NaN,NaN,-0.078615,-0.031524,NaN,NaN,NaN,0
4,2020-01-08,73.017853,73.318893,71.565636,71.565636,132079200,NaN,NaN,72.298413,72.313661,0.016087,NaN,NaN,-0.004881,-0.026196,NaN,NaN,NaN,0


In [22]:
# Create Target Again
df['Target'] = (
    df['Close'].shift(-1)
    > df['Close']
).astype(int)

df = df.dropna()

print(df.shape)
print(df.isnull().sum())

(1457, 20)
Date            0
Close           0
High            0
Low             0
Open            0
Volume          0
SMA_20          0
SMA_50          0
EMA_20          0
EMA_50          0
Daily_Return    0
Volatility      0
RSI             0
MACD            0
Signal_Line     0
BB_Middle       0
BB_Upper        0
BB_Lower        0
Signal          0
Target          0
dtype: int64


In [23]:
features = [
    'Close',
    'Volume',
    'SMA_20',
    'SMA_50',
    'EMA_20',
    'EMA_50',
    'Daily_Return',
    'Volatility',
    'RSI',
    'MACD',
    'Signal_Line'
]

X = df[features]
y = df['Target']

In [24]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(np.isnan(X_scaled).sum())

0


In [25]:
'Normalize Data LSTMs require scaled inputs.'
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(1457, 11)


In [26]:
X_seq = []
y_seq = []

lookback = 10

for i in range(lookback, len(X_scaled)):
    X_seq.append(X_scaled[i-lookback:i])
    y_seq.append(y.iloc[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print(X_seq.shape)

(1447, 10, 11)


In [27]:
print(X_seq.shape)
print(y_seq.shape)

(1447, 10, 11)
(1447,)


In [28]:
'''Chronological Split

Since this is a time-series problem, we again use chronological splitting.'''
split = int(len(X_seq) * 0.8)

X_train = X_seq[:split]
X_test = X_seq[split:]

y_train = y_seq[:split]
y_test = y_seq[split:]

LSTM Model:

In [29]:
model = Sequential([
    Input(shape=(10,11)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

In [30]:
# Compile:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [31]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.4987 - loss: 0.6944 - val_accuracy: 0.5414 - val_loss: 0.6902
Epoch 2/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5272 - loss: 0.6925 - val_accuracy: 0.5345 - val_loss: 0.6925
Epoch 3/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5290 - loss: 0.6922 - val_accuracy: 0.5414 - val_loss: 0.6914
Epoch 4/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5315 - loss: 0.6921 - val_accuracy: 0.5414 - val_loss: 0.6901
Epoch 5/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5246 - loss: 0.6908 - val_accuracy: 0.5414 - val_loss: 0.6921
Epoch 6/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5272 - loss: 0.6920 - val_accuracy: 0.5414 - val_loss: 0.6900
Epoch 7/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5272 - loss: 0.6908 - val_accuracy: 0.5414 - val_loss: 0.6914
Epoch 8/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5333 - loss: 0.6920 - val_accuracy: 0.5414 - v